# BloodMNIST Experiments (CNN or MLP)
Use this notebook as the runner. Shared code lives in the `ml/` package.

In [10]:
from ml.data import build_bloodmnist_dataloaders
from ml.models import CNN, MLP
from ml.setup import get_device, make_criterion, make_optimizer
from ml.training import EarlyStopping, train_with_validation, evaluate_accuracy

In [11]:
BATCH_SIZE = 128
data_bundle = build_bloodmnist_dataloaders(batch_size=BATCH_SIZE, download=True)

train_loader = data_bundle['train_loader']
val_loader = data_bundle['val_loader']
test_loader = data_bundle['test_loader']
n_channels = data_bundle['n_channels']
n_classes = data_bundle['n_classes']

print(f'n_channels={n_channels}, n_classes={n_classes}')

n_channels=3, n_classes=8


In [12]:
MODEL_NAME = 'mlp'   # choose: 'cnn' or 'mlp'
OPTIMIZER_NAME = 'adam'  # choose: 'adam' or 'sgd'
LEARNING_RATE = 0.0003  # 0.01 for sgd
MOMENTUM = 0.9
DROPOUT = 0.3
PATIENCE = 30
EARLY_STOPIING_DELTA = 0.001
WEIGHT_DECAY = 0.0005

device = get_device()

if MODEL_NAME == 'cnn':
    model = CNN(in_channels=n_channels, num_classes=n_classes, dropout=DROPOUT).to(device)
elif MODEL_NAME == 'mlp':
    model = MLP(input_dim=n_channels * 28 * 28, num_classes=n_classes, dropout=DROPOUT).to(device)
else:
    raise ValueError('MODEL_NAME must be cnn or mlp')

criterion = make_criterion('cross_entropy')
optimizer = make_optimizer(
    model,
    name=OPTIMIZER_NAME,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
early_stopping = EarlyStopping(PATIENCE, EARLY_STOPIING_DELTA)

print(f'Device: {device}')
print(f'Model: {MODEL_NAME}')
print(f'Optimizer: {OPTIMIZER_NAME}, lr={LEARNING_RATE}')

Device: cpu
Model: mlp
Optimizer: adam, lr=0.0003


In [13]:
history = train_with_validation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=350,
    early_stopping=early_stopping,
)

print('Stopped epoch:', history['stopped_epoch'])
print('Best epoch:', history['best_epoch'])

Epoch 1	Training Loss: 1.5560	Validation Loss: 1.1430
Epoch 2	Training Loss: 1.0931	Validation Loss: 0.9255
Epoch 3	Training Loss: 0.9474	Validation Loss: 0.8073
Epoch 4	Training Loss: 0.8850	Validation Loss: 0.7456
Epoch 5	Training Loss: 0.8349	Validation Loss: 0.8268
Epoch 6	Training Loss: 0.8041	Validation Loss: 0.6779
Epoch 7	Training Loss: 0.7766	Validation Loss: 0.6507
Epoch 8	Training Loss: 0.7827	Validation Loss: 0.6999
Epoch 9	Training Loss: 0.7237	Validation Loss: 0.6182
Epoch 10	Training Loss: 0.7061	Validation Loss: 0.6033
Epoch 11	Training Loss: 0.6876	Validation Loss: 0.5877
Epoch 12	Training Loss: 0.6807	Validation Loss: 0.5769
Epoch 13	Training Loss: 0.6407	Validation Loss: 0.5748
Epoch 14	Training Loss: 0.6406	Validation Loss: 0.5652
Epoch 15	Training Loss: 0.6209	Validation Loss: 0.5435
Epoch 16	Training Loss: 0.6236	Validation Loss: 0.5696
Epoch 17	Training Loss: 0.6145	Validation Loss: 0.5866
Epoch 18	Training Loss: 0.5985	Validation Loss: 0.5322
Epoch 19	Training L

KeyboardInterrupt: 

In [ ]:
test_accuracy = evaluate_accuracy(model, test_loader, device)
print(f'Test accuracy: {test_accuracy:.2f}%')

Test accuracy: 88.40%
